# Semana 14: Manifest, hashes y tracking

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
RANDOM_STATE = 42
output_dir = Path("artifacts_semana14")
output_dir.mkdir(exist_ok=True)

In [3]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev, y_train_dev, test_size=0.25, stratify=y_train_dev, random_state=RANDOM_STATE
)
print({"train": X_train.shape, "dev": X_dev.shape, "test": X_test.shape})

{'train': (341, 30), 'dev': (114, 30), 'test': (114, 30)}


In [4]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
pipeline.fit(X_train, y_train)

def classification_metrics(X_part, y_part):
    probability = pipeline.predict_proba(X_part)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    return {
        "f1": float(f1_score(y_part, prediction)),
        "roc_auc": float(roc_auc_score(y_part, probability)),
    }, probability

metrics_dev, proba_dev = classification_metrics(X_dev, y_dev)
metrics_test, proba_test = classification_metrics(X_test, y_test)
metrics = {"dev": metrics_dev, "test": metrics_test}
metrics

{'dev': {'f1': 0.993006993006993, 'roc_auc': 0.9980347199475925},
 'test': {'f1': 0.9861111111111112, 'roc_auc': 0.9950396825396824}}

In [5]:
model_path = output_dir / "model.joblib"
metrics_path = output_dir / "metrics.json"
dump(pipeline, model_path)
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(model_path, metrics_path)

artifacts_semana14/model.joblib artifacts_semana14/metrics.json


In [6]:
import platform
import sklearn

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "params": {
        "random_state": RANDOM_STATE,
        "split": {"test_size": 0.2, "dev_fraction_of_train_dev": 0.25, "stratified": True},
        "model": "LogisticRegression",
        "threshold": 0.5,
    },
    "metrics": metrics,
    "artifacts": {path.name: file_sha256(path) for path in [model_path, metrics_path]},
}
manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest

{'environment': {'python': '3.12.0',
  'numpy': '1.26.4',
  'pandas': '2.3.3',
  'scikit_learn': '1.9.0'},
 'params': {'random_state': 42,
  'split': {'test_size': 0.2,
   'dev_fraction_of_train_dev': 0.25,
   'stratified': True},
  'model': 'LogisticRegression',
  'threshold': 0.5},
 'metrics': {'dev': {'f1': 0.993006993006993, 'roc_auc': 0.9980347199475925},
  'test': {'f1': 0.9861111111111112, 'roc_auc': 0.9950396825396824}},
 'artifacts': {'model.joblib': 'b9050de5733a010bb9a0527ec726a1197eac4b42a10b59e5519ada7900ede4c9',
  'metrics.json': '1474af369096c9e91f0547fe7b856160ba0dd6c47b4291ade6d4e0d8e8fe19ee'}}

## Cierre

El manifest identifica la corrida; su hash demuestra identidad de bytes, no corrección científica.

In [7]:
loaded_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
required = {"environment", "params", "metrics", "artifacts"}
assert required <= set(loaded_manifest)
hash_checks = {name: file_sha256(output_dir / name) == digest for name, digest in loaded_manifest["artifacts"].items()}
hash_checks

{'model.joblib': True, 'metrics.json': True}

In [8]:
from joblib import load

reloaded = load(model_path)
repeated = reloaded.predict_proba(X_test)[:, 1]
{"max_probability_difference": float(np.max(np.abs(repeated - proba_test))), "same_hash": file_sha256(model_path) == manifest["artifacts"][model_path.name]}

{'max_probability_difference': 0.0, 'same_hash': True}